<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Statistical-Inference-Finite-Epidemics/blob/main/Sim3_SI_for_Epidemic_Risk_Functionals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# SIMULATION 3 — FAST CPU SCREENING WITH CALIBRATION DIAGNOSTICS (v3)
# Single Colab cell, no external Python file needed; NumPy, SciPy, pandas, Numba.
# Working-likelihood + linearized-score multiplier approximation, NOT exact hidden-CTMC MLE.
# Integer N is selected by local discrete profiling, and its uncertainty is approximated
# via a continuous-N relaxed sandwich/influence approximation. Validation uses full refits.
# NOT certified 95% inference under misspecification or discrete N.
# Separate joint-relaxed-N and conditional-N cases; empirical oracle maxT check.
# IMPORTANT: model/estimation are unchanged in principle; this is a diagnostic redesign.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import sys, json, math, pickle, time, pathlib, hashlib, datetime, warnings, traceback
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln, digamma
from scipy.stats import chi2, beta as beta_dist
from numba import njit

BASE_CFG=dict(
    R0=60, I0=1, N_min=125, N_max=250, N_local=2,
    MC=500, B=999, m_values=[50,100,200,500],
    mechanisms=['CTMC','NB_control'], reporting=[.75,.90],
    scenario_multipliers={'Main':[1.0,1.0],'Perturbed':[1.10,.95]},
    q12=.35, q21=.25, regime_factors=[1.2,.8], alpha=.05,
    seed=20261011, maxiter=55, optimizer_starts=2, checkpoint_every=10,
    validate=True, validate_B=199, validate_MC=2, model_check_outbreaks=1000,
    min_valid_boot_fraction=.90,
    log_bounds=[[float(np.log(.04)),float(np.log(3.0))],
                [float(np.log(.035)),float(np.log(1.5))],
                [float(np.log(.12)),float(np.log(50.0))]],
    covariance_min_eigen=1e-7, min_contrast_se=1e-9,
    N_edge_expand=2, profile_audit=True, profile_audit_step=10,
)
INTERVENTIONS={
 'No intervention':(1,1,1,1,1,1),
 'Weak control':(.98,1.03,.98,.98,.90,1.10),
 'Broad response':(.78,.86,1.10,1.06,1.30,.80),
 'High-regime suppression':(.50,1.22,1.15,.92,.75,1.35),
 'Low-regime stabilization':(1.15,.40,.92,1.18,1.75,.42),
 'Switching-focused':(1.06,.88,.96,1.05,3.60,.25),
 'Recovery-focused':(1.08,1.10,1.55,1.28,1,1),
 'Emergency response':(.62,1.04,1.30,.96,2.15,.38),
}
NAMES=list(INTERVENTIONS)
K=len(NAMES)
CFG=BASE_CFG.copy()
RNG=np.random.default_rng(CFG['seed'])
DUR=None


@njit

def _deriv(y,b,g,N,mode,q0,q1,fac1,fac2,p0,p1):

    if mode==0:

        rate=b*y[0]*y[1]/N

        return np.array([-rate,rate-g*y[1]])

    f0=b*fac1*y[0]*y[2]/(N*p0)

    f1=b*fac2*y[1]*y[3]/(N*p1)

    return np.array([-f0-q0*y[0]+q1*y[1],

                     -f1+q0*y[0]-q1*y[1],

                     f0-g*y[2]-q0*y[2]+q1*y[3],

                     f1-g*y[3]+q0*y[2]-q1*y[3]])

@njit

def _expected(beta,gamma,N,durations,mode,report,R0,I0,q0,q1,fac1,fac2):

    S0=N-R0-I0

    p0=q1/(q0+q1);p1=1-p0

    if mode==0: y=np.array([float(S0),float(I0)])

    else: y=np.array([p0*S0,p1*S0,p0*I0,p1*I0])

    out=np.zeros(len(durations))

    for j in range(len(durations)):

        initial=y[0] if mode==0 else y[0]+y[1]

        steps=max(2,int(np.ceil(durations[j]*8)))

        h=durations[j]/steps

        for k in range(steps):

            k1=_deriv(y,beta,gamma,N,mode,q0,q1,fac1,fac2,p0,p1)

            k2=_deriv(y+.5*h*k1,beta,gamma,N,mode,q0,q1,fac1,fac2,p0,p1)

            k3=_deriv(y+.5*h*k2,beta,gamma,N,mode,q0,q1,fac1,fac2,p0,p1)

            k4=_deriv(y+h*k3,beta,gamma,N,mode,q0,q1,fac1,fac2,p0,p1)

            y=y+h*(k1+2*k2+2*k3+k4)/6

        after=y[0] if mode==0 else y[0]+y[1]

        out[j]=max(1e-12,(initial-after)*report)

    return out

@njit
def _simulate_outbreaks_jit(m,beta,gamma,N,report,mode,times,seed,R0,I0,q12,q21,fac1,fac2):
    # Exact Gillespie CTMC; fast, seeded, observed only at weekly reported counts.
    np.random.seed(seed)
    out=np.zeros((m,len(times)),dtype=np.int32)
    p0=q21/(q12+q21)
    max_time=times[-1]
    for j in range(m):
        S=N-R0-I0; I=I0; phase=0 if np.random.random()<p0 else 1
        t=0.0
        while I>0 and t<max_time:
            b=beta*(fac1 if phase==0 else fac2) if mode==1 else beta
            inf=b*S*I/N
            rec=gamma*I
            sw=(q12 if phase==0 else q21) if mode==1 else 0.0
            rate=inf+rec+sw
            if rate<=0:break
            t+=-np.log(max(1e-300,np.random.random()))/rate
            if t>=max_time: break
            x=np.random.random()*rate
            if x<inf:
                S-=1;I+=1
                if np.random.random()<report:
                    k=np.searchsorted(times,t,side='right')
                    if k<len(times):out[j,k]+=1
            elif x<inf+rec:
                I-=1
            else:
                phase=1-phase
    return out

@njit

def _tail_risk_compiled(b1,b2,g1,g2,q12,q21,N,R0,I0,thresholds,pi0):

    M=N-R0; s0=M-I0

    prev=np.zeros((M+1,s0+1,2))

    for i in range(M+1):

        prev[i,0,0]=1.0;prev[i,0,1]=1.0

    for s in range(1,s0+1):

        curr=np.zeros((M+1,s0+1,2))

        curr[0,0,0]=1.0;curr[0,0,1]=1.0

        for i in range(1,M-s+1):

            inf0=b1*s*i/N; inf1=b2*s*i/N

            rec0=g1*i;rec1=g2*i

            a0=inf0+rec0+q12;a1=inf1+rec1+q21

            determinant=a0*a1-q12*q21

            for n in range(s+1):

                r0=rec0*curr[i-1,n,0]

                r1=rec1*curr[i-1,n,1]

                if n>0:

                    r0+=inf0*prev[i+1,n-1,0]

                    r1+=inf1*prev[i+1,n-1,1]

                curr[i,n,0]=(a1*r0+q12*r1)/determinant

                curr[i,n,1]=(q21*r0+a0*r1)/determinant

        prev=curr

    p0=pi0;p1=1-p0

    pmf=np.empty(s0+1)

    for n in range(s0+1):

        pmf[n]=p0*prev[I0,n,0]+p1*prev[I0,n,1]

    ret=np.zeros(len(thresholds))

    for k in range(len(thresholds)):

        c=thresholds[k]

        for n in range(c+1,s0+1):ret[k]+=pmf[n]

    return ret,pmf.sum()

def mean_incidence(beta,gamma,N,durations,mode='single',pi_report=1.0):
    return _expected(float(beta),float(gamma),float(N),np.asarray(durations,dtype=np.float64),
        0 if mode=='single' else 1,float(pi_report),CFG['R0'],CFG['I0'],
        CFG['q12'],CFG['q21'],CFG['regime_factors'][0],CFG['regime_factors'][1])


def simulate_outbreaks(m,beta,gamma,N,report,mode='two'):
    seed=int(RNG.integers(1,2**31-1))
    return _simulate_outbreaks_jit(int(m),float(beta),float(gamma),int(N),float(report),
        1 if mode=='two' else 0,np.cumsum(DUR),seed,CFG['R0'],CFG['I0'],
        CFG['q12'],CFG['q21'],CFG['regime_factors'][0],CFG['regime_factors'][1])


def tail_risk(b1,b2,g1,g2,q12,q21,N,thresholds):
    ret,mass=_tail_risk_compiled(float(b1),float(b2),float(g1),float(g2),float(q12),float(q21),
        int(N),CFG['R0'],CFG['I0'],np.asarray(thresholds,dtype=np.int64),
        CFG['q21']/(CFG['q12']+CFG['q21']))
    if abs(mass-1)>1.e-6 or not np.isfinite(ret).all():
        raise ArithmeticError('CTMC final-size distribution does not normalize')
    return ret


def risk_matrix(est, thresholds):
    beta,gamma,phi,N=est
    N=int(round(N))
    b=np.array(CFG['regime_factors'])*beta
    ret=[]
    for mult in INTERVENTIONS.values():
        a=np.asarray(mult,dtype=float)
        ret.append(tail_risk(b[0]*a[0],b[1]*a[1],gamma*a[2],gamma*a[3],
                             CFG['q12']*a[4],CFG['q21']*a[5],N,thresholds))
    return np.asarray(ret)


def initial_data():
    if (run_dir/'cameroon_weekly_used.csv').exists():
        file=run_dir/'cameroon_weekly_used.csv'
    else:
        dirs=[pathlib.Path.cwd(),pathlib.Path('/content'),ROOT,pathlib.Path('/mnt/data')]
        files=[]
        for folder in dirs:
            if folder.exists():
                files += sorted(folder.glob('Cameroon_weekly_incidence*.csv'))
        if not files and 'google.colab' in sys.modules:
            from google.colab import files as google_files
            print('Upload your original Cameroon_weekly_incidence.csv now')
            uploaded=google_files.upload()
            if uploaded: files=[pathlib.Path(next(iter(uploaded)))]
        if not files:
            raise FileNotFoundError('Upload Cameroon_weekly_incidence.csv (24 weeks).')
        file=files[0]
    df=pd.read_csv(file,encoding='utf-8-sig')
    df.columns=[str(v).strip().lower() for v in df.columns]
    if not {'week_ending','observed_days'}.issubset(df.columns):
        raise ValueError('Expected week_ending and observed_days in weekly CSV')
    col='reported_cases' if 'reported_cases' in df else 'observed_cases'
    if col not in df: raise ValueError('Expected reported_cases or observed_cases')
    df['week_ending']=pd.to_datetime(df['week_ending'])
    df=df.sort_values('week_ending').reset_index(drop=True)
    out=pd.DataFrame({'week_ending':df['week_ending'],
                      'observed_days':pd.to_numeric(df['observed_days']),
                      'reported_cases':pd.to_numeric(df[col])})
    if len(out)!=24: warnings.warn(f'Expected 24 weeks; got {len(out)}')
    if (out.observed_days<=0).any() or (out.reported_cases<0).any():
        raise ValueError('Invalid duration/case count')
    file_hash=hashlib.sha256(out.to_csv(index=False).encode()).hexdigest()
    if state.get('data_hash') and state['data_hash']!=file_hash:
        raise ValueError('Input data differs from checkpoint; do not resume.')
    out.to_csv(run_dir/'cameroon_weekly_used.csv',index=False)
    state['data_hash']=file_hash
    return out.reported_cases.to_numpy(dtype=np.int64),out.observed_days.to_numpy(dtype=float)/7


def observation_stats(Y):
    Y=np.asarray(Y,dtype=np.int64)
    if Y.ndim==1: Y=Y[None,:]
    m,T=Y.shape
    values,freq=np.unique(Y,return_counts=True)
    return m,Y.sum(axis=0,dtype=float),values.astype(float),freq.astype(float)


def nll(z,N,stats,mode,report):
    if N<=CFG['R0']+CFG['I0'] or N<CFG['N_min'] or N>CFG['N_max']:
        return 1e90
    beta,gamma,phi=np.exp(np.clip(z,-100,100))
    mu=mean_incidence(beta,gamma,N,DUR,mode,report)
    if not np.isfinite(mu).all() or np.any(mu<=0): return 1e90
    m,ysum,values,freq=stats
    logsum=np.log(phi+mu)
    ll=float(np.dot(freq,gammaln(values+phi))-len(DUR)*m*gammaln(phi))
    ll+=m*phi*np.sum(np.log(phi)-logsum)
    ll+=np.dot(ysum,np.log(mu)-logsum)
    penalty=.5*((z[1]-np.log(1/3))/.177)**2
    ans=-ll+penalty
    return float(ans) if np.isfinite(ans) else 1e90


def fit_joint_fast(Y,mode,report,start=None,calibration=False):
    """Same penalized NB objective, continuous relaxation then LOCAL integer profile."""
    stats=observation_stats(Y)
    bounds=CFG['log_bounds']+[(CFG['N_min'],CFG['N_max'])]
    if start is None:
        starts=[np.r_[np.log([.65,.33,2.5]),(CFG['N_min']+CFG['N_max'])/2],
                np.r_[np.log([1.1,.25,7.]),(CFG['N_min']+CFG['N_max'])/2]]
    else:
        starts=[np.r_[np.log(np.maximum(start[:3],1.e-8)),float(start[3])]]
    opts={'maxiter':CFG['maxiter'],'ftol':1e-6}
    fits=[]
    for st in starts[:(CFG['optimizer_starts'] if start is None else 1)]:
        fit=minimize(lambda x:nll(x[:3],x[3],stats,mode,report),st,
                     method='L-BFGS-B',bounds=bounds,options=opts)
        if np.isfinite(fit.fun) and fit.fun<1e80: fits.append(fit)
    if not fits: raise ValueError('continuous fitting failed')
    best=min(fits,key=lambda f:f.fun)
    middle=int(np.clip(round(best.x[3]),CFG['N_min'],CFG['N_max']))
    radius=int(CFG['N_local'])
    candidates=set(range(max(CFG['N_min'],middle-radius),min(CFG['N_max'],middle+radius)+1))
    if start is not None: candidates.add(int(start[3]))
    if calibration:
        candidates.update(range(CFG['N_min'],CFG['N_max']+1,20))
        candidates.add(CFG['N_max'])
    results=[]
    for n in sorted(candidates):
        f=minimize(lambda z:nll(z,n,stats,mode,report),best.x[:3],
                  method='L-BFGS-B',bounds=CFG['log_bounds'],options=opts)
        if np.isfinite(f.fun) and f.fun<1e80:
            results.append((float(f.fun),n,f.x))
    if not results: raise ValueError('integer profile fitting failed')
    # If the local integer optimum hits the profile-window edge, widen the
    # search before accepting it. This is still not an exhaustive N search.
    extra=0
    for _ in range(int(CFG.get('N_edge_expand',2))):
        score,n0,z0=min(results,key=lambda a:a[0])
        new=[]
        for ntest in (n0-1,n0+1):
            if CFG['N_min']<=ntest<=CFG['N_max'] and ntest not in candidates:
                new.append(ntest)
        if not new: break
        for ntest in new:
            f=minimize(lambda z:nll(z,ntest,stats,mode,report),z0,
                       method='L-BFGS-B',bounds=CFG['log_bounds'],options=opts)
            candidates.add(ntest)
            if np.isfinite(f.fun) and f.fun<1e80:
                results.append((float(f.fun),ntest,f.x));extra+=1
    loss,n,z=min(results,key=lambda a:a[0])
    estimated=np.r_[np.exp(z),float(n)]
    diagnostic={'N_near_profile_edge':int(abs(n-middle)>=radius),
                'N_cont':float(best.x[3]),
                'N_optimizer_boundary':int(n in (CFG['N_min'],CFG['N_max'])),
                'N_candidate_count':len(candidates),
                'N_extra_profile_fits':extra,
                'N_profile_best_loss':float(loss),
                'N_profile_low':int(min(candidates)),
                'N_profile_high':int(max(candidates))}
    return estimated,loss,diagnostic


def NB_scores_hessian(Y,fit,report,mode):
    """Per-outbreak score, 4d observed Hessian in (log b,log gamma,log phi,N/100).
    N is RELAXED CONTINUOUS at this step; this is not an exact discrete-N bootstrap.
    Includes the original gamma penalty in the Hessian, but not in each outbreak score.
    """
    beta,gamma,phi,N=fit
    T=Y.shape[1]
    w=np.array([np.log(beta),np.log(gamma),np.log(phi),N/100],dtype=float)
    def mean_at(x):
        return mean_incidence(np.exp(x[0]),np.exp(x[1]),100*x[3],DUR,mode,report)
    mu=mean_at(w)
    dm=np.empty((T,4))
    for j,h in enumerate([.0008,.0008,0.,.001]):
        if j==2:
            dm[:,j]=0
            continue
        hi=w.copy();lo=w.copy()
        hi[j]+=h;lo[j]-=h
        if j==3:
            hi[j]=min(hi[j],CFG['N_max']/100)
            lo[j]=max(lo[j],CFG['N_min']/100)
        dm[:,j]=(mean_at(hi)-mean_at(lo))/(hi[j]-lo[j])
    y=np.asarray(Y,dtype=float)
    common=y/mu[None,:]-(y+phi)/(phi+mu)[None,:]
    S=common@dm
    psi=digamma(y+phi)-digamma(phi)
    phiscore=phi*np.sum(psi +np.log(phi/(phi+mu))[None,:] +1-(phi+y)/(phi+mu)[None,:],axis=1)
    S[:,2]=phiscore
    S=S-S.mean(axis=0,keepdims=True)
    if not np.isfinite(S).all(): raise ValueError('nonfinite NB outbreak scores')

    stats=observation_stats(Y)
    steps=np.array([.004,.004,.006,.008],dtype=float)
    def obj(x): return nll(x[:3],100*x[3],stats,mode,report)
    f0=obj(w)
    H=np.zeros((4,4))
    for j in range(4):
        h=steps[j]
        xp=w.copy();xm=w.copy();xp[j]+=h;xm[j]-=h
        if j==3 and (xp[j]>CFG['N_max']/100 or xm[j]<CFG['N_min']/100):
            # At a boundary: numerical Hessian is unreliable. Flag instead.
            raise ValueError('N at search boundary: nonregular for 4D influence expansion')
        H[j,j]=(obj(xp)-2*f0+obj(xm))/h**2
        for k in range(j+1,4):
            hp=steps[k]
            vals=[]
            for sj,sk in [(1,1),(1,-1),(-1,1),(-1,-1)]:
                xx=w.copy();xx[j]+=sj*h;xx[k]+=sk*hp
                vals.append(obj(xx))
            H[j,k]=(vals[0]-vals[1]-vals[2]+vals[3])/(4*h*hp)
            H[k,j]=H[j,k]
    H=(H+H.T)/2
    eig,U=np.linalg.eigh(H)
    positive=max(eig.max(),1.e-8)
    minimum=max(CFG['covariance_min_eigen'],positive*1e-7)
    bad=int(np.sum(eig<=minimum))
    if bad>=1: raise ValueError('Non-positive / weak 4D Hessian: joint-N approximation unreliable')
    Hinv=(U*(1/np.maximum(eig,minimum)))@U.T
    cov=Hinv@(S.T@S)@Hinv
    cov=(cov+cov.T)/2
    if not np.isfinite(cov).all(): raise ValueError('invalid sandwich covariance')
    return S,Hinv,cov,{'hessian_min_eig':float(eig.min()),
                       'hessian_reg_eigs':bad,'N_se_relaxed':float(100*np.sqrt(max(0,cov[3,3])))}



def NB_conditional_N_sandwich(Y,fit,report,mode):
    """Fallback: conditional on fitted integer N ONLY. No joint-N coverage claim."""
    beta,gamma,phi,N=fit
    z=np.log([beta,gamma,phi])
    mu=mean_incidence(beta,gamma,N,DUR,mode,report)
    dm=np.empty((len(DUR),2))
    for j in range(2):
        zp=z.copy();zm=z.copy();h=.0008
        zp[j]+=h;zm[j]-=h
        a=mean_incidence(np.exp(zp[0]),np.exp(zp[1]),N,DUR,mode,report)
        b=mean_incidence(np.exp(zm[0]),np.exp(zm[1]),N,DUR,mode,report)
        dm[:,j]=(a-b)/(2*h)
    y=np.asarray(Y,dtype=float)
    common=y/mu[None,:]-(y+phi)/(phi+mu)[None,:]
    S=np.zeros((len(Y),3));S[:,:2]=common@dm
    S[:,2]=phi*np.sum(digamma(y+phi)-digamma(phi)+np.log(phi/(phi+mu))[None,:]
                   +1-(phi+y)/(phi+mu)[None,:],axis=1)
    S-=S.mean(axis=0)
    stats=observation_stats(Y)
    def obj(zz):return nll(zz,N,stats,mode,report)
    f0=obj(z);hs=np.array([.004,.004,.006]);H=np.zeros((3,3))
    for j in range(3):
        zp=z.copy();zm=z.copy();zp[j]+=hs[j];zm[j]-=hs[j]
        H[j,j]=(obj(zp)-2*f0+obj(zm))/hs[j]**2
        for k in range(j+1,3):
            vals=[]
            for sj,sk in [(1,1),(1,-1),(-1,1),(-1,-1)]:
                zz=z.copy();zz[j]+=sj*hs[j];zz[k]+=sk*hs[k]
                vals.append(obj(zz))
            H[j,k]=(vals[0]-vals[1]-vals[2]+vals[3])/(4*hs[j]*hs[k])
            H[k,j]=H[j,k]
    eig,U=np.linalg.eigh((H+H.T)/2)
    floor=max(1e-7,float(max(eig.max(),1e-8))*1e-7)
    if (eig<=floor).any():raise ValueError('Conditional-N Hessian not positive definite')
    inv=(U*(1/np.maximum(eig,floor)))@U.T
    S4=np.c_[S,np.zeros(len(S))]
    H4=np.zeros((4,4));H4[:3,:3]=inv
    return S4,H4,dict(inference_scope='conditional_N',N_se_relaxed=np.nan,
                      hessian_min_eig=float(eig.min()),hessian_reg_eigs=int((eig<=floor).sum()))

def risk_jacobian(est,thresholds):
    """Exact CTMC recursion at reference + perturbations, numerical risk derivatives.
    phi has zero derivative. N gradient is local finite difference between integer models.
    """
    baseline=risk_matrix(est,thresholds)
    J=np.zeros(baseline.shape+(4,))
    for j in [0,1]:
        up=est.copy();dn=est.copy();h=.002
        up[j]*=math.exp(h);dn[j]*=math.exp(-h)
        J[:,:,j]=(risk_matrix(up,thresholds)-risk_matrix(dn,thresholds))/(2*h)
    n=int(round(est[3]))
    l=max(CFG['N_min'],n-1);u=min(CFG['N_max'],n+1)
    if u==l: raise ValueError('integer N perturbation failed')
    up=est.copy();dn=est.copy();up[3]=u;dn[3]=l
    J[:,:,3]=100*(risk_matrix(up,thresholds)-risk_matrix(dn,thresholds))/(u-l)
    return baseline,J


def nondominated(D):
    """D[l,k,t]: upper bound on R_l-R_k. All thresholds must certify dominance."""
    retained=set(range(K))
    for k in range(K):
        for l in range(K):
            if k!=l and np.all(D[l,k]<=1e-12) and np.any(D[l,k]<-1e-9):
                retained.discard(k)
                break
    return retained


def oracle_screen(R):
    return nondominated(R[:,None,:]-R[None,:,:])


def order_q(vals,alpha):
    vals=np.asarray(vals,dtype=float)
    if len(vals)==0 or not np.isfinite(vals).all(): raise ValueError('invalid bootstrap sample')
    rank=min(len(vals),max(1,math.ceil((len(vals)+1)*(1-alpha))))
    return float(np.partition(vals,rank-1)[rank-1])


def bound_screen(Dhat,bootdiff,alpha,studentized=True):
    """Global one-sided bootstrap across all ordered intervention pairs and thresholds.
    bootdiff is an array of risk DIFFERENCES from the fitted Dhat, B x K x K x T.
    Exact deterministic zero contrasts remain zero, rather than +inf.
    """
    offdiag=~np.eye(K,dtype=bool)
    dev=-bootdiff[:,offdiag,:]  # (B, ordered-pairs,T)
    sd=np.std(dev,axis=0,ddof=1)
    deterministic=(sd<1e-12)&(abs(Dhat[offdiag])<1e-12)
    unstable=(sd<CFG['min_contrast_se'])&(~deterministic)
    if studentized:
        scales=np.maximum(sd,CFG['min_contrast_se'])
        valid=~deterministic & ~unstable
        stat=np.max(dev[:,valid]/scales[valid],axis=1) if np.any(valid) else np.zeros(len(dev))
        q=order_q(stat,alpha)
        up=Dhat[offdiag]+q*scales
    else:
        valid=~deterministic & ~unstable
        stat=np.max(dev[:,valid],axis=1) if np.any(valid) else np.zeros(len(dev))
        q=order_q(stat,alpha)
        up=Dhat[offdiag]+q
    up[deterministic]=0.0
    up[unstable]=np.inf
    U=np.full_like(Dhat,np.inf)
    U[offdiag]=up
    return nondominated(U),U,q,int(unstable.sum()),int(deterministic.sum())


def fast_multiplier_screen(fit,Y,report,thresholds,B=999):
    """Global one-sided cluster score multiplier.

    Computes a joint relaxed-N approximation when its Hessian is regular;
    otherwise returns CONDITIONAL-N diagnostics only. Never conflate these.
    """
    failure_reason=''
    try:
        S,Hinv,C,diag=NB_scores_hessian(Y,fit,report,'two')
        if diag['N_se_relaxed']>0.5*(CFG['N_max']-CFG['N_min']):
            raise ValueError('Relaxed-N uncertainty exceeds half of N search range')
        diag['inference_scope']='joint_N_linearized'
    except ValueError as exc:
        failure_reason=str(exc)
        S,Hinv,diag=NB_conditional_N_sandwich(Y,fit,report,'two')
    diag['joint_N_failure_reason']=failure_reason
    R,J=risk_jacobian(fit,thresholds)
    if diag['inference_scope']=='conditional_N':
        J[:,:,-1]=0.0
    Dhat=R[:,None,:]-R[None,:,:]
    multipliers=(2*RNG.integers(0,2,size=(B,len(Y)),dtype=np.int8)-1).astype(float)
    # Outbreak-level scores preserve within-outbreak weekly dependence.
    dw=(multipliers@S)@Hinv.T
    Jdiff=J[:,None,:,:]-J[None,:,:,:]
    shifts=np.einsum('bi,lkti->blkt',dw,Jdiff,optimize=True)
    global_set,U,q,nunstable,ndeterministic=bound_screen(Dhat,shifts,CFG['alpha'],True)
    env_set,Uraw,qraw,_,_=bound_screen(Dhat,shifts,CFG['alpha'],False)
    # For testing the calibration we use the *same* standardization as in
    # bound_screen and evaluate the actual MC maximum against q_student.
    off=~np.eye(K,dtype=bool)
    sdev=np.std(shifts[:,off,:],axis=0,ddof=1)
    exactzero=(sdev<1e-12)&(np.abs(Dhat[off])<1e-12)
    unstable=(sdev<CFG['min_contrast_se']) & ~exactzero
    scales=np.maximum(sdev,CFG['min_contrast_se'])
    diag.update(q_student=float(q),q_raw=float(qraw),unstable_contrasts=nunstable,
                deterministic_contrasts=ndeterministic,
                min_positive_contrast_sd=float(np.min(sdev[~exactzero])) if (~exactzero).any() else np.nan)
    return dict(PlugIn=oracle_screen(R),GlobalMaxT=global_set,
                CalibratedEnvelope=env_set,upper_student=U,upper_raw=Uraw,
                diagnostics=diag,Rhat=R,scales=scales,zero_mask=exactzero,
                unstable_mask=unstable)


def oracle_max_statistic(fast,truth_R):
    """Simulation-only diagnostic (uses the oracle, never to construct bands)."""
    Dt=truth_R[:,None,:]-truth_R[None,:,:]
    Dr=fast['Rhat'][:,None,:]-fast['Rhat'][None,:,:]
    off=~np.eye(K,dtype=bool)
    diff=Dt[off]-Dr[off]
    mask=~fast['zero_mask'] & ~fast['unstable_mask']
    stat=float(np.max(diff[mask]/fast['scales'][mask])) if mask.any() else 0.0
    return stat,float(np.max(diff[mask])) if mask.any() else 0.0


def fit_global_N_audit(Y,mode,report,start,step=10):
    """Slow, *selected MC only* coarse-to-fine independent integer-N profile.

    Returns the best discovered N and the loss improvement over the fast local
    fit. Nonconvexity means it is an audit, not a proof of global optimality.
    """
    stats=observation_stats(Y)
    zn=np.log(np.maximum(start[:3],1e-10))
    base=[CFG['N_min'],CFG['N_max'],int(start[3])]
    base.extend(range(CFG['N_min'],CFG['N_max']+1,int(step)))
    profile={}
    opts={'maxiter':CFG['maxiter'],'ftol':1e-6}
    for n in sorted(set(base)):
        f=minimize(lambda z:nll(z,n,stats,mode,report),zn,
                   method='L-BFGS-B',bounds=CFG['log_bounds'],options=opts)
        if np.isfinite(f.fun) and f.fun<1e80:
            profile[n]=(float(f.fun),f.x)
    if not profile: raise RuntimeError('global N profile audit failed')
    nbest=min(profile,key=lambda n:profile[n][0])
    for n in range(max(CFG['N_min'],nbest-int(step)),min(CFG['N_max'],nbest+int(step))+1):
        if n in profile: continue
        f=minimize(lambda z:nll(z,n,stats,mode,report),profile[nbest][1],
                   method='L-BFGS-B',bounds=CFG['log_bounds'],options=opts)
        if np.isfinite(f.fun) and f.fun<1e80:profile[n]=(float(f.fun),f.x)
    nbest=min(profile,key=lambda n:profile[n][0]); bestloss=profile[nbest][0]
    local_loss=nll(np.log(np.maximum(start[:3],1e-10)),start[3],stats,mode,report)
    return dict(N_profile_audit_best=int(nbest),N_profile_audit_gap=float(local_loss-bestloss),
                N_profile_audit_Nchanged=float(nbest!=int(start[3])),
                N_profile_audit_candidates=len(profile))

def full_boot_validation(Y,fit,report,mechanism,thresholds,fit_R,B):
    """Actual full refit/bootstrap, integer N re-selected for each draw.
    Validation only: this is the expensive reference that main study approximates.
    """
    risks=[]; failed=0; params=[]
    for b in range(B):
        try:
            if mechanism=='CTMC':
                simulated=simulate_outbreaks(len(Y),fit[0],fit[1],fit[3],report,'two')
            else:
                mu=mean_incidence(fit[0],fit[1],fit[3],DUR,'two',report)
                simulated=RNG.negative_binomial(fit[2],fit[2]/(fit[2]+mu),size=Y.shape)
            bp,_,_=fit_joint_fast(simulated,'two',report,start=fit)
            risks.append(risk_matrix(bp,thresholds))
            params.append(bp)
        except (ValueError,RuntimeError,ArithmeticError,FloatingPointError,np.linalg.LinAlgError):
            failed+=1
        if (b+1)%15==0: checkpoint(force=True,quiet=True)
    if len(risks)<max(10,math.ceil(B*CFG['min_valid_boot_fraction'])):
        raise ValueError(f'only {len(risks)}/{B} valid full refits')
    B_R=np.asarray(risks)
    Dhat=fit_R[:,None,:]-fit_R[None,:,:]
    Dboot=B_R[:,:,None,:]-B_R[:,None,:,:]
    changes=Dboot-Dhat[None,:,:,:]
    glob,U,q,unstable,det=bound_screen(Dhat,changes,CFG['alpha'],True)
    env,Uraw,qraw,_,_=bound_screen(Dhat,changes,CFG['alpha'],False)
    return dict(GlobalMaxT=glob,CalibratedEnvelope=env,upper_student=U,upper_raw=Uraw,
                q_student=q,q_raw=qraw,n_boot_valid=len(risks),n_boot_failed=failed,
                N_sd_full=float(np.std(np.asarray(params)[:,3],ddof=1)))


def save_pickle(path,obj):
    tmp=path.with_suffix('.tmp')
    with tmp.open('wb') as f:
        pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)


def checkpoint(force=False,quiet=False):
    global last_save,run_clock
    t=time.monotonic()
    if not force and t-last_save<CFG['checkpoint_seconds']:
        return
    state['elapsed']+=t-run_clock
    run_clock=time.monotonic()
    state['rng_state']=RNG.bit_generator.state
    save_pickle(run_dir/'checkpoint.pkl',state)
    last_save=time.monotonic()
    if not quiet:
        print(f'[CHECKPOINT] complete={len(state["rows"])} elapsed={state["elapsed"]/3600:.3f} h',flush=True)


def summarize():
    if not state['rows']: return
    df=pd.DataFrame(state['rows'])
    df.to_csv(run_dir/'main_replicates.csv',index=False)
    keys=['mechanism','environment','report','m']
    metrics=[x for x in df.columns if x.endswith(('_false_elim','_power','_retained','_contrast_cover'))]
    metrics += [x for x in ['N_hat','N_error','N_relaxed_se','b_bias','gamma_bias',
                'N_profile_edge','regularized_Hessian','q_student','q_raw',
                'unstable_contrasts','deterministic_contrasts','oracle_maxT','oracle_max_raw',
                'N_profile_audit_gap','N_profile_audit_Nchanged','time_seconds'] if x in df]
    valid=df[df.status=='OK']
    if len(valid):
        tb=valid.groupby(keys,sort=False)[metrics].mean().reset_index()
        tb['valid_MC']=valid.groupby(keys,sort=False).size().values
        total=df.groupby(keys,sort=False).size().reset_index(name='attempted_MC')
        tb=tb.merge(total,on=keys,how='left')
        tb['failed_MC']=tb.attempted_MC-tb.valid_MC
        conditional=valid.groupby(keys,sort=False)['inference_scope'].agg(
            lambda x:float(np.mean(np.asarray(x)=='conditional_N')))
        tb['conditional_N_fraction']=conditional.values
        # Coverage is separately reported for genuine 4D relaxed-N fits.
        jrows=valid[valid.inference_scope=='joint_N_linearized']
        jgroup=jrows.groupby(keys,sort=False)
        joint=jgroup.agg(joint_N_MC=('status','size'),
                          joint_N_contrast_coverage=('GlobalMaxT_contrast_cover','mean'))
        tb=tb.merge(joint.reset_index(),on=keys,how='left')
        tb['joint_N_MC']=tb['joint_N_MC'].fillna(0).astype(int)
        tb['joint_N_coverage_MCSE']=np.sqrt(
            tb['joint_N_contrast_coverage']*(1-tb['joint_N_contrast_coverage'])/
            tb['joint_N_MC'].replace(0,np.nan))
        for meth in ['PlugIn','GlobalMaxT','CalibratedEnvelope']:
            col=meth+'_false_elim'
            tt=valid.groupby(keys,sort=False)[col].agg(['sum','count'])
            ub=[beta_dist.ppf(.95,1+int(s),int(n)-int(s)) if s<n else 1.0 for s,n in tt.to_numpy()]
            tb[meth+'_false_elim_upper95']=ub
        tb.to_csv(run_dir/'table1_fast_screening.csv',index=False)
        # One row per scenario and inference scope: never silently average
        # conditional-N approximations with relaxed joint-N approximations.
        byscope=valid.groupby(keys+['inference_scope'],sort=False)[
            ['GlobalMaxT_contrast_cover','GlobalMaxT_false_elim',
             'GlobalMaxT_power','N_error','b_bias','gamma_bias',
             'q_student','oracle_maxT']].agg(['mean','count'])
        byscope.to_csv(run_dir/'table1_by_inference_scope.csv')
        readable=valid.groupby(keys+['inference_scope'],sort=False).agg(
            valid_MC=('status','size'), contrast_coverage=('GlobalMaxT_contrast_cover','mean'),
            false_elim=('GlobalMaxT_false_elim','mean'), power=('GlobalMaxT_power','mean'),
            N_error=('N_error','mean'),q_mean=('q_student','mean'))
        readable.to_csv(run_dir/'table1_scope_summary.csv')
        print('\nSCREENING BY INFERENCE SCOPE (DIAGNOSTIC, NOT CERTIFIED 95%):\n',
              readable.round(4).to_string(),flush=True)
        # Empirical 95%-quantile of the TRUE normalized max error, only
        # measurable in simulations. Compare with mean bootstrap q.
        cal=valid.groupby(keys+['inference_scope'],sort=False).agg(
            n=('status','size'), q_boot_mean=('q_student','mean'),
            q_oracle_95=('oracle_maxT',lambda x:float(np.quantile(x,.95))),
            contrast_coverage=('GlobalMaxT_contrast_cover','mean'))
        cal.to_csv(run_dir/'table2_maxT_calibration.csv')
        print(f'[EXPORT] {len(valid)} valid MC; see table1_scope_summary.csv and table2_maxT_calibration.csv',flush=True)
    df[df.status!='OK'].to_csv(run_dir/'failures.csv',index=False)
    if state['validation']:
        v=pd.DataFrame(state['validation']);v.to_csv(run_dir/'validation_full_bootstrap.csv',index=False)
        if 'screen_disagreement' in v:
            v.groupby(keys,sort=False)[['screen_disagreement','q_full','q_multiplier','seconds']].mean().to_csv(run_dir/'validation_comparison.csv')


def choose_validation(mechanism,env,report,m,rep):
    if not CFG['validate'] or rep>=CFG['validate_MC']: return False
    # Four prespecified validation configurations, two data mechanisms.
    return ((env=='Main' and report==.75 and m in (50,500)) or
            (env=='Perturbed' and report==.90 and m in (50,500)))


def apply_stats(row,prefix,method,oracle,dominated,true_diff,offdiag):
    retained=method[prefix]
    row[prefix+'_false_elim']=float(not oracle.issubset(retained))
    row[prefix+'_power']=float(len(dominated & (set(range(K))-retained))/max(1,len(dominated)))
    row[prefix+'_retained']=int(len(retained))
    if prefix=='GlobalMaxT':
        row[prefix+'_contrast_cover']=float(np.all(true_diff[offdiag]<=method['upper_student'][offdiag]+1e-12))
    if prefix=='CalibratedEnvelope':
        row[prefix+'_contrast_cover']=float(np.all(true_diff[offdiag]<=method['upper_raw'][offdiag]+1e-12))


def run_experiment():
    global run_clock, last_save
    if state['calibration'] is None:
        cal,score,diag=fit_joint_fast(OBS,'single',1.,calibration=True)
        state['calibration']=dict(parameters=cal,score=score,diagnostic=diag)
        print('[CALIBRATION]',cal,'penalized loss:',score,flush=True)
        checkpoint(force=True)
    cal=state['calibration']['parameters']
    scl=int(cal[3])-CFG['R0']-CFG['I0']
    thresholds=np.unique(np.rint(np.array([.20,.30,.40,.55])*scl).astype(np.int64))
    print('[THRESHOLDS] fixed absolute:',thresholds,'| N reference:',cal[3],flush=True)
    if 'CTMC' in CFG['mechanisms'] and not state.get('adequacy_done'):
        checks=[]
        for env,factors in CFG['scenario_multipliers'].items():
            for report in CFG['reporting']:
                beta,gamma=cal[0]*factors[0],cal[1]*factors[1]
                y=simulate_outbreaks(CFG['model_check_outbreaks'],beta,gamma,cal[3],report,'two')
                observed=y.mean(axis=0)
                working=mean_incidence(beta,gamma,cal[3],DUR,'two',report)
                rel=float(np.sum(abs(observed-working))/max(1e-10,observed.sum()))
                checks.append(dict(environment=env,report=report,relative_L1=rel,
                                   observed_total=float(observed.sum()),working_total=float(working.sum())))
                print(f'[ADEQUACY] {env},p={report:.2f}, relative L1={rel:.3f}',flush=True)
        pd.DataFrame(checks).to_csv(run_dir/'working_mean_adequacy.csv',index=False)
        state['adequacy_done']=True
        checkpoint(force=True)
    jobs=[(mech,env,report,m) for mech in CFG['mechanisms']
          for env in CFG['scenario_multipliers']
          for report in CFG['reporting'] for m in CFG['m_values']]
    total=len(jobs)*CFG['MC']
    for mechanism,env,report,m in jobs:
        factors=CFG['scenario_multipliers'][env]
        truth=np.array([cal[0]*factors[0],cal[1]*factors[1],cal[2],cal[3]])
        R_true=risk_matrix(truth,thresholds)
        oracle=oracle_screen(R_true)
        dominated=set(range(K))-oracle
        Dt=R_true[:,None,:]-R_true[None,:,:]
        offdiag=~np.eye(K,dtype=bool)
        done=sum(r['mechanism']==mechanism and r['environment']==env and
                 r['report']==report and r['m']==m for r in state['rows'])
        print(f'[SCENARIO] {mechanism} {env} p={report}, m={m} complete={done}/{CFG["MC"]},'
              f' oracle_nondominated={len(oracle)}',flush=True)
        for j in range(done,CFG['MC']):
            clock=time.monotonic()
            key=(mechanism,env,report,m,j)
            active=state['active']
            if active is not None and tuple(active['key'])!=key:
                raise RuntimeError('active checkpoint has unexpected scenario key')
            try:
                if active is None:
                    if mechanism=='CTMC':
                        Y=simulate_outbreaks(m,truth[0],truth[1],truth[3],report,'two')
                    else:
                        mu=mean_incidence(truth[0],truth[1],truth[3],DUR,'two',report)
                        Y=RNG.negative_binomial(truth[2],truth[2]/(truth[2]+mu),size=(m,len(DUR)))
                    fitted,loss,fitdiag=fit_joint_fast(Y,'two',report)
                    fast=fast_multiplier_screen(fitted,Y,report,thresholds,CFG['B'])
                    active=dict(key=key,Y=Y,fit=fitted,fitdiag=fitdiag,fast=fast,
                                validation_complete=False,started_elapsed=state['elapsed']+time.monotonic()-run_clock)
                    state['active']=active
                    checkpoint(force=True,quiet=True)
                else:
                    Y=active['Y'];fitted=active['fit'];fitdiag=active['fitdiag'];fast=active['fast']
                row=dict(mechanism=mechanism,environment=env,report=report,m=m,rep=j,status='OK')
                row['oracle_maxT'],row['oracle_max_raw']=oracle_max_statistic(fast,R_true)
                for meth in ['PlugIn','GlobalMaxT','CalibratedEnvelope']:
                    apply_stats(row,meth,fast,oracle,dominated,Dt,offdiag)
                row.update(N_hat=float(fitted[3]),N_error=float(abs(fitted[3]-truth[3])),
                           N_relaxed_se=fast['diagnostics']['N_se_relaxed'],
                           b_bias=float(fitted[0]-truth[0]),gamma_bias=float(fitted[1]-truth[1]),
                           N_profile_edge=fitdiag['N_near_profile_edge'],
                           inference_scope=fast['diagnostics']['inference_scope'],
                           regularized_Hessian=fast['diagnostics']['hessian_reg_eigs'],
                           q_student=fast['diagnostics']['q_student'],q_raw=fast['diagnostics']['q_raw'],
                           unstable_contrasts=fast['diagnostics']['unstable_contrasts'],
                           deterministic_contrasts=fast['diagnostics']['deterministic_contrasts'])
                if choose_validation(mechanism,env,report,m,j):
                    if CFG.get('profile_audit',True):
                        if 'profile_audit' not in active:
                            try:
                                active['profile_audit']=fit_global_N_audit(
                                    Y,'two',report,fitted,step=CFG['profile_audit_step'])
                            except Exception as exc:
                                active['profile_audit']={'profile_audit_error':str(exc)[:160]}
                        row.update(active['profile_audit'])
                    if not active['validation_complete']:
                        st=time.monotonic()
                        try:
                            full=full_boot_validation(Y,fitted,report,mechanism,thresholds,
                                                      fast['Rhat'],CFG['validate_B'])
                            vrow=dict(mechanism=mechanism,environment=env,report=report,m=m,rep=j,
                                      status='OK',q_full=full['q_student'],
                                      q_multiplier=fast['diagnostics']['q_student'],
                                      screen_disagreement=float(full['GlobalMaxT']!=fast['GlobalMaxT']),
                                      full_false_elim=float(not oracle.issubset(full['GlobalMaxT'])),
                                      full_contrast_cover=float(np.all(Dt[offdiag]<=full['upper_student'][offdiag]+1e-12)),
                                      n_boot_valid=full['n_boot_valid'],n_boot_failed=full['n_boot_failed'],
                                      N_boot_sd=full['N_sd_full'],
                                      full_contrast_violations=int(np.sum(
                                          Dt[offdiag]>full['upper_student'][offdiag]+1e-12)),
                                      fast_contrast_violations=int(np.sum(
                                          Dt[offdiag]>fast['upper_student'][offdiag]+1e-12)),
                                      N_fast_est=float(fitted[3]),
                                      N_profile_audit_gap=float(active.get('profile_audit',{}).get('N_profile_audit_gap',np.nan)),
                                      inference_scope=fast['diagnostics']['inference_scope'])
                        except Exception as exc:
                            vrow=dict(mechanism=mechanism,environment=env,report=report,m=m,rep=j,
                                      status='FAILED',reason=str(exc)[:180])
                        vrow['seconds']=time.monotonic()-st
                        state['validation'].append(vrow)
                        active['validation_complete']=True
                        checkpoint(force=True,quiet=True)
                row['time_seconds']=time.monotonic()-clock
            except (ValueError,RuntimeError,ArithmeticError,FloatingPointError,np.linalg.LinAlgError) as exc:
                row=dict(mechanism=mechanism,environment=env,report=report,m=m,rep=j,
                         status='FAILED',reason=str(exc)[:180],time_seconds=time.monotonic()-clock)
            state['rows'].append(row)
            state['active']=None
            if len(state['rows'])%CFG['checkpoint_every']==0: checkpoint(force=True,quiet=True)
            if len(state['rows'])<=4 or len(state['rows'])%20==0:
                avg=state['elapsed']+(time.monotonic()-run_clock)
                eta=(total-len(state['rows']))*avg/max(1,len(state['rows']))/3600
                print(f'[MC] {len(state["rows"])}/{total}, time={avg/3600:.2f}h,'
                      f' linear ETA={eta:.1f}h (later m may be slower)',flush=True)
        summarize()
        checkpoint(force=True)
    summarize();checkpoint(force=True)
    print('\nDONE. See CSV outputs:',run_dir)


def startup():
    global CFG,RNG,ROOT,run_dir,state,DUR,OBS,run_clock,last_save
    try:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
        ROOT=pathlib.Path('/content/drive/MyDrive/Epidemic_Risk_Sim3_FastMultiplier_Diagnostics_v3')
    except ImportError:
        ROOT=pathlib.Path(os.environ.get('SIM3_OUTPUT','./Epidemic_Risk_Sim3_FastMultiplier_Diagnostics_v3'))
    ROOT.mkdir(parents=True,exist_ok=True)
    mode=os.environ.get('SIM3_MODE')
    if mode not in ('1','2'):
        mode=input('\nSIM 3 FAST CPU: 1 = START NEW | 2 = RESUME latest run: ').strip()
    if mode not in ('1','2'): raise ValueError('Choose 1 or 2')
    latest=ROOT/'LATEST.txt'
    if mode=='1':
        run_kind=os.environ.get('SIM3_RUN')
        if run_kind not in ('1','2','3'):
            run_kind=input('1 = tiny smoke test; 2 = pilot (30 MC); 3 = full (500 MC): ').strip()
        if run_kind not in ('1','2','3'): raise ValueError('Choose 1, 2 or 3')
        CFG=json.loads(json.dumps(BASE_CFG))
        if run_kind=='1':
            CFG.update(MC=2,B=49,validate_B=19,validate_MC=1,model_check_outbreaks=50,
                       m_values=[50],mechanisms=['CTMC'],reporting=[.75],
                       scenario_multipliers={'Main':[1.,1.]})
        if run_kind=='2':
            CFG.update(MC=30,B=199,validate_B=49,validate_MC=2,model_check_outbreaks=200)
        mechanism=os.environ.get('SIM3_MECHANISM')
        if run_kind=='3' and mechanism not in ('1','2'):
            mechanism=input('Run 1=CTMC only (4,000 MC) or 2=CTMC+NB control (8,000 MC) [2]: ').strip() or '2'
        if run_kind=='3' and mechanism=='1': CFG['mechanisms']=['CTMC']
        ROOT.mkdir(parents=True,exist_ok=True)
        run_dir=ROOT/('run_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        run_dir.mkdir(parents=True,exist_ok=False)
        latest.write_text(str(run_dir))
        state={'rows':[],'validation':[],'active':None,'calibration':None,
               'elapsed':0.,'rng_state':np.random.default_rng(CFG['seed']).bit_generator.state,
               'config_hash':hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()}
        (run_dir/'config.json').write_text(json.dumps(CFG,indent=2))
    else:
        if not latest.exists(): raise FileNotFoundError('No latest run found')
        run_dir=pathlib.Path(latest.read_text().strip())
        CFG=json.loads((run_dir/'config.json').read_text())
        with (run_dir/'checkpoint.pkl').open('rb') as f:state=pickle.load(f)
        expected=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()
        if state['config_hash']!=expected: raise RuntimeError('config changed since checkpoint')
        print('[RESUME] completed',len(state['rows']),'from',run_dir)
    RNG=np.random.default_rng();RNG.bit_generator.state=state['rng_state']
    OBS,DUR=initial_data()
    run_clock=time.monotonic();last_save=run_clock
    if mode=='1':checkpoint(force=True,quiet=True)
    print('Experiment:',run_dir)
    print('Design:',CFG['mechanisms'],'MC=',CFG['MC'],'B=',CFG['B'], 'm=',CFG['m_values'])
    try:
        run_experiment()
    except KeyboardInterrupt:
        checkpoint(force=True);summarize()
        print('Interrupted. Re-run the SAME notebook cell and select 2=RESUME.');raise
    except Exception:
        checkpoint(force=True);summarize()
        traceback.print_exc();raise

if __name__=='__main__':
    startup()


Mounted at /content/drive

SIM 3 FAST CPU: 1 = START NEW | 2 = RESUME latest run: 1
1 = tiny smoke test; 2 = pilot (30 MC); 3 = full (500 MC): 2
Upload your original Cameroon_weekly_incidence.csv now


Saving Cameroon_weekly_incidence.csv to Cameroon_weekly_incidence.csv
Experiment: /content/drive/MyDrive/Epidemic_Risk_Sim3_FastMultiplier_Diagnostics_v3/run_20261012_092918_588623
Design: ['CTMC', 'NB_control'] MC= 30 B= 199 m= [50, 100, 200, 500]
[CALIBRATION] [  0.83928585   0.33463878   4.79437929 163.        ] penalized loss: 9.095704669741998
[CHECKPOINT] complete=0 elapsed=0.001 h
[THRESHOLDS] fixed absolute: [20 31 41 56] | N reference: 163.0
[ADEQUACY] Main,p=0.75, relative L1=1.371
[ADEQUACY] Main,p=0.90, relative L1=1.028
[ADEQUACY] Perturbed,p=0.75, relative L1=1.286
[ADEQUACY] Perturbed,p=0.90, relative L1=1.366
[CHECKPOINT] complete=0 elapsed=0.001 h
[SCENARIO] CTMC Main p=0.75, m=50 complete=0/30, oracle_nondominated=1
[MC] 1/960, time=0.00h, linear ETA=2.5h (later m may be slower)
[MC] 2/960, time=0.00h, linear ETA=1.7h (later m may be slower)
[MC] 3/960, time=0.00h, linear ETA=1.1h (later m may be slower)
[MC] 4/960, time=0.00h, linear ETA=0.9h (later m may be slower)
